In [2]:
import numpy as np
print("numpy:", np.__version__)

import gymnasium as gym
print("gymnasium:",gym.__version__)

try:
    import tensorflow as tf
    print("tensorflow:", tf.__version__)
except ImportError:
    print("tensorflow: NOT installed")

numpy: 1.26.4
gymnasium: 1.2.3
tensorflow: 2.18.0


"CartPole-v1" is an environment where a pole is balanced on a cart and the goal is to prevent the pole from falling.

In [3]:
import gymnasium as gym
import numpy as np

# Create the environment  
env = gym.make('CartPole-v1')

# Set random seed for reproducibility  
np.random.seed(42)
env.reset(seed=42)

(array([ 0.0273956 , -0.00611216,  0.03585979,  0.0197368 ], dtype=float32),
 {})

### Defining DQN model


- `Sequential` model: a linear stack of layers in Keras. 
- `Dense` layers: fully connected layers.  
- `input_dim`: the size of the input layer, corresponding to the state size.  
- `activation='relu'`: Rectified Linear Unit activation function.  
- `activation='linear'`: linear activation function for the output layer. 
- `Adam` optimizer: an optimization algorithm that adjusts the learning rate based on gradients.


In [5]:
# suppress warnings for a cleaner notebook or console experience
import warnings
warnings.filterwarnings('ignore')

# disable warnings for a cleaner notebook or console experience
def warn(*args, **kwargs):
    pass
warnings.warn = warn

# import necessary libraries
import gymnasium as gym
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam

env = gym.make("CartPole-v1")

def build_model(state_size, action_size):
    model = Sequential()
    model.add(Input(shape=(state_size,)))
    model.add(Dense(24, activation='relu'))
    model.add(Dense(24, activation='relu'))
    model.add(Dense(action_size, activation='linear'))
    model.compile(loss='mse', optimizer=Adam(learning_rate=0.001))
    return model

state_size = int(env.observation_space.shape[0])
action_size = int(env.action_space.n)

model = build_model(state_size, action_size)
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 24)             │           120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │           600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 2)              │            50 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 770 (3.01 KB)

 Trainable params: 770 (3.01 KB)

 Non-trainable params: 0 (0.00 B)

### Implement after replay buffer

Replay buffer stores agent's experiences for training. I will make it with "deque".<br>

Each stored item has:
- state       = what the environment looked like before action
- action      = what the agent did
- reward      = what reward it got
- next_state  = what the environment looked like after action
- done        = whether the episode ended

In [8]:
from collections import deque
import random

memory = deque(maxlen=2000)
def remember(state, action, reward, next_state, done):
    memory.append((state, action, reward, next_state, done)) # saving those inputs into memory as one tuple

### epsilon-greedy policy

Balancing exploration and exploitation by choosing random actions with probability epsilon.

- `epsilon`: exploration rate.  
- `epsilon_min`: minimum exploration rate.  
- `epsilon_decay`: decay rate for epsilon after each episode.  
- `act()`: chooses an action based on the epsilon-greedy policy.

In [9]:
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.995
 
def act(state):
    if np.random.rand() <= epsilon:
        return random.randrange(action_size)
    q_values = model.predict(state)
    return np.argmax(q_values[0])

### Q-learning update
Implement Q-learning update for training DQN learning<br>

- `replay()`: samples a random minibatch from memory and trains the model.  
- `target`: the Q-value target, which is updated using the reward and the maximum Q-value of the next state.  
- `model.fit()`: trains the model on the updated Q-values.


In [10]:
def replay(batch_size):
    global epsilon
    minibatch = random.sample(memory, batch_size)
    for state, action, reward, next_state, done in minibatch:
        target = reward
        if not done:
            target = reward + gamma * np.amax(model.predict(next_state)[0])
        target_f = model.predict(state)
        target_f[0][action] = target
        model.fit(state, target_f, epochs=1, verbose=0)
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

Train DQN learning

